# Main Pipeline Scan Planning

This notebook documents how individual observing sessions of the Lab 4 galactic-plane HI survey are *formed*: from the abstract sampling grid down to the ordered list of cells the Leuschner dish actually visits during a single window of operating hours. The driver script `scripts/main/main` is config-only and delegates all planning to `scripts/main/_survey.py`. The exposition here mirrors that logic, makes the theoretical assumptions explicit, and renders representative coverage from real archived sessions in both galactic ($\ell, b$) and topocentric (alt, az) coordinates.

## 1. Motivation

The science targets in `lab_dish/HI1.tex` (galactic-plane rotation curve, plane warp, North Polar Spur, NCP supershell, etc.) all require *maps*: dense sampling of the 21 cm line over hundreds of square degrees in galactic coordinates. The smallest such project lists $\sim 400$ pointings; the largest, $\sim 10\,000$ pointings. At our Leuschner integration time of $\sim 60$ s on-source per cell (set by SNR requirements at the dish's nominal $T_{\rm sys} \sim 200$ K and the bandwidth-shifted frequency-switching scheme), a full survey requires tens of hours of dish time spread over multiple nights.

Two facts constrain how that time is spent.

1. **Cells are only accessible for part of each sidereal day.** A given $(\ell, b)$ traces an alt/az curve set by Leuschner's latitude ($+37.92^\circ$). Cells north of the celestial equator dip behind the dish's lower altitude limit ($17^\circ$); cells very near the celestial pole bump against the upper limit ($83^\circ$); the azimuth cable wrap forbids a wedge near az$=0/360^\circ$. Cells that are visible *now* will not be visible *in three hours*, and vice versa.

2. **The dish is slow.** A typical inter-cell slew + per-cell schedule (ABBA frequency switch, four cal dumps plus eight obs dumps) totals $\approx 95$ s. A drift-monitoring recal pair (slew $\to$ recal field $\to$ slew back) runs every ten cells and burns $\approx 305$ s. Once a cell is committed to the queue, several minutes elapse before the dish reaches it; alt/az drifts $\sim 1^\circ$ in that window.

Together these mean we cannot statically partition the survey into sessions. Each session must be planned *forward* from the moment the script starts: pick the cells the dish will still be able to reach when the scan order brings the executor to them, and skip the ones the sky will have rotated out of by then. The rest of this notebook walks through that planning, then visualises what comprehensive sessions look like on the ground.

In [ ]:
import math
import re
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.collections import LineCollection, PolyCollection
import astropy.units as u
from astropy.coordinates import SkyCoord, ICRS

import ugradiolab.plotting as plotting
from ugradiolab.astronomy import LEO_LAT_DEG, LEO_LON_DEG

import plotters
from plotters import (
    add_never_observable_overlay,
    add_topo_inaccessible_overlay,
    beam_outline_polys,
)
from utils.timing_stats import load as load_timing_stats

%matplotlib inline

# --- Survey config (mirrors scripts/main/main : SurveyConfig) -----------
L_CENTER = 120.0
L_MIN, L_MAX = -60.0, 300.0
B_MIN, B_MAX = -20, 20
B_STEP = 2
PHYSICAL_SPACING_DEG = 2.0

# Telescope limits
MIN_ALT_DEG = 17.0
MAX_ALT_DEG = 83.0
AZ_MIN_DEG, AZ_MAX_DEG = 7.0, 348.0

# Beam (HPBW ~ 3.4 deg at 1.4 GHz on the 4.5 m dish)
HPBW_DEG = 3.4

# Forward-sim time budget
RECAL_BLOCK_SEC = 305.0
RECAL_EVERY_N = 10
MAX_PLAN_HORIZON_H = 4.0

stats = load_timing_stats(
    'artifacts/main_timing_stats.json',
    archive_dir='data/archive/main',
)
CELL_TIME_P50 = stats['cell_total_time_sec']['p50']
print(f"timing stats: {stats['n_cells_observed']} cells across "
      f"{stats['n_sessions']} archived sessions")
print(f"  intra-cell cadence: p50 {stats['intra_cell_cadence_sec']['p50']:.1f}s")
print(f"  slew gap:           p50 {stats['slew_gap_sec']['p50']:.1f}s")
print(f"  cell total:         p50 {CELL_TIME_P50:.1f}s")
print(f"  duty cycle:         {stats['duty_cycle']:.2f}")

## 2. The galactic-coordinate sampling grid

The HI1 instructions (\S 8 in the LaTeX source) prescribe Nyquist sampling at *two samples per FWHM* in great-circle distance. With our $4.5$ m dish's HPBW of $\approx 3.4^\circ$ at $1.42$ GHz, that yields a $2^\circ$ angular spacing.

A naive $(\Delta \ell, \Delta b) = (2^\circ, 2^\circ)$ grid oversamples in $\ell$ at high $|b|$: the great-circle distance between two cells at fixed $\ell$-step is $\Delta \ell \cdot \cos(b)$, so near the poles a $2^\circ$ step is much smaller than the beam. We correct this with $\Delta \ell = 2^\circ / \cos(b)$ at each $b$ row, which preserves uniform great-circle sampling and reduces total cell count substantially toward higher $|b|$.

A second refinement is the **brick-interleave**: two phases of the grid, even ($b \in \{-20, -18, \ldots, 20\}$ at $\ell$ centered on $\ell_0 = 120^\circ$) and odd ($b \in \{-19, -17, \ldots, 19\}$ with $\ell$ offset by half a step). Stacked, the two phases approximate a hexagonal sampling pattern: adjacent rows are offset by half a beam, so reconstructed maps have lower aliasing than a square grid. We currently run only the even phase (`phases=('even',)` in the driver) until baseline coverage is complete; the odd-phase entrypoint is preserved for the second pass.

Cells are stored in *column-major* order: contiguous runs at fixed $\ell$ swept through $b$, zig-zagged so successive cells share an azimuth. This minimises slew time relative to a row-major sweep that would otherwise jump across the full $\ell$ range on every row break.

In [ ]:
def build_l_row(b_deg, l_center=L_CENTER, l_min=L_MIN, l_max=L_MAX):
    """Cells at fixed b with cos(b)-corrected longitude spacing."""
    cos_b = math.cos(math.radians(b_deg))
    if cos_b <= 0:
        return [l_center] if l_min <= l_center <= l_max else []
    dl = PHYSICAL_SPACING_DEG / cos_b
    l_vals = [round(l_center, 2)]
    l = l_center + dl
    while l <= l_max:
        l_vals.append(round(l, 2))
        l += dl
    l = l_center - dl
    while l >= l_min:
        l_vals.append(round(l, 2))
        l -= dl
    return sorted(v for v in l_vals if l_min <= v <= l_max)


def build_galplane_grid(phase='even'):
    if phase == 'even':
        b_vals = list(range(B_MIN, B_MAX + 1, B_STEP))
    else:
        b_vals = list(range(B_MIN + 1, B_MAX, B_STEP))
    all_cells = []
    for b in b_vals:
        if phase == 'odd':
            half_step = PHYSICAL_SPACING_DEG / (2 * math.cos(math.radians(b)))
            l_center = L_CENTER + half_step
        else:
            l_center = L_CENTER
        for l in build_l_row(b, l_center=l_center):
            all_cells.append((l, b))
    if not all_cells:
        return []
    all_cells.sort(key=lambda c: c[0])
    col_tol = PHYSICAL_SPACING_DEG / 2
    columns = [[all_cells[0]]]
    for cell in all_cells[1:]:
        if cell[0] - columns[-1][0][0] <= col_tol:
            columns[-1].append(cell)
        else:
            columns.append([cell])
    out = []
    for col_idx, col in enumerate(columns):
        col_sorted = sorted(col, key=lambda c: c[1])
        if col_idx % 2 == 1:
            col_sorted = list(reversed(col_sorted))
        for row_idx, (l, b) in enumerate(col_sorted):
            out.append((col_idx, row_idx, l, b))
    return out

even_grid = build_galplane_grid('even')
odd_grid = build_galplane_grid('odd')
print(f'even phase: {len(even_grid)} cells')
print(f'odd phase:  {len(odd_grid)} cells')
print(f'combined:   {len(even_grid) + len(odd_grid)} cells')

## 3. Telescope-induced sky accessibility

Before forward-simulating any particular session, it helps to look at which cells are *ever* reachable at all. The static never-observable mask sweeps hour-angle through a full sidereal day and asks: does there exist any HA at which this $(\ell, b)$ satisfies

$$17^\circ \le \mathrm{alt} \le 83^\circ \quad \text{and} \quad 7^\circ \le \mathrm{az} \le 348^\circ ?$$

Cells that fail at every HA are *permanently* invisible from Leuschner. These are the deep southern declinations (cells whose maximum altitude is below $17^\circ$, i.e.\ $\delta \lesssim -35^\circ$) and a thin circumpolar cap at $\delta \gtrsim +89^\circ$ that stays above $83^\circ$. The remaining cells are visible for at least one window per sidereal day, but the windows are not synchronous with each other.

The two panels below show this mask in galactic coordinates and in topocentric coordinates. The same red-hatched style is used throughout the lab's plotters to indicate "no data possible here".

In [ ]:
fig = plt.figure(figsize=(plotting.TEXTWIDTH_IN, 5.5))

ax_g = fig.add_subplot(2, 1, 1, projection='mollweide')
ax_g.grid(True, **{k: v for k, v in plotting.GRID_STYLE.items() if k != 'color'},
          color=plotting.NEUTRAL_COLOR)
add_never_observable_overlay(ax_g, center_l=L_CENTER,
                             latitude_deg=LEO_LAT_DEG,
                             min_alt_deg=MIN_ALT_DEG, max_alt_deg=MAX_ALT_DEG,
                             az_min_deg=AZ_MIN_DEG, az_max_deg=AZ_MAX_DEG)
l_line = np.linspace(-np.pi, np.pi, 500)
ax_g.plot(l_line, np.zeros_like(l_line), lw=plotting.LW_FINE,
          color='k', alpha=plotting.ALPHA_FAINT)
tick_locs = np.arange(-150, 180, 30)
ax_g.set_xticks(np.deg2rad(tick_locs))
ax_g.set_xticklabels(
    [rf'{int((t + L_CENTER) % 360)}$^\circ$' for t in tick_locs],
    fontsize=plotting.TICK_SIZE - 3,
)
ax_g.set_title(r'Galactic ($\ell$, $b$), centred on $\ell=120^\circ$',
               fontsize=plotting.LABEL_SIZE)

ax_t = fig.add_subplot(2, 1, 2, projection='mollweide')
ax_t.grid(True, **{k: v for k, v in plotting.GRID_STYLE.items() if k != 'color'},
          color=plotting.NEUTRAL_COLOR)
add_topo_inaccessible_overlay(ax_t)
az_ticks = np.array([-150, -120, -90, -60, -30, 0, 30, 60, 90, 120, 150])
ax_t.set_xticks(np.deg2rad(az_ticks))
ax_t.set_xticklabels(
    [rf'{int(t % 360)}$^\circ$' for t in az_ticks],
    fontsize=plotting.TICK_SIZE - 3,
)
ax_t.set_title(r'Topocentric (az, alt), centred on az$=0/360^\circ$',
               fontsize=plotting.LABEL_SIZE)

fig.tight_layout()
plt.show()

## 4. Forming a session by forward simulation

A session is the cell list produced by *one* invocation of `plan_phase` in `_survey.py`. The driver loops `plan_phase` $\to$ execute forever; each top-of-loop call rebuilds the plan against the current wallclock, so cells naturally roll in as they rise and out as they set.

Within one call the planning logic is:

1. **Build** the full grid for the active phase.
2. **Filter by az side.** Each cell's instantaneous az at $t=t_0$ classifies it as *rising* (az $\in [7^\circ, 180^\circ]$) or *setting* (az $\in (180^\circ, 348^\circ]$). Side selection picks whichever side has more *currently accessible* cells. This avoids the cable-wrap penalty in `Beware Operating near Azimuth 0/360` (HI1.tex \S 7.2): the dish never crosses the $0/360$ seam within a session.
3. **Drop already-complete cells** via the completeness index (a per-LO, per-kind dump counter built from the on-disk `data/main/session_*/<cell>/*.npz` filenames). Cells whose dump counts already meet the schedule's target are skipped; cells listed in `artifacts/main_reobserve.json` are forced in regardless.
4. **Forward-simulate.** Walk the surviving cells in column-major order. For each cell $i$ project its observation time as

$$t_i = t_0 + (i + 0.5) \, t_\text{cell} + n_\text{recal}(i) \, t_\text{recal},$$

where $t_\text{cell}$ is the empirically measured p50 per-cell duration (from `timing_stats`), $t_\text{recal} = 305$ s is the p90 of an end-to-end recal cycle, and $n_\text{recal}(i) = 1 + \lfloor i / 10 \rfloor$ accounts for the start-of-session warm-up plus injected drift checks every $10$ cells. Evaluate alt/az at $t_i$ via the cached ICRS $\to$ Leuschner closed-form transform (matches astropy to $< 1^\circ$ everywhere, much faster) and keep the cell only if it still passes the alt/az limits.
5. **Horizon cut.** Cells whose projected $t_i - t_0 > 4$ hours are dropped: anything that far out will be re-planned by the next top-of-loop iteration anyway, and committing to it would let runtime drift accumulate.
6. **Strategy selection.** Steps 4--5 are repeated for the four column-major sweep variants (l ascending/descending $\times$ b ascending/descending) on both az sides; the variant with the most survivors wins.

The output of `plan_phase` is the *cell list*: an ordered sequence of $(\ell, b)$ that the executor will visit, interleaved at runtime with recal cells from `RECAL_TARGETS`.

In [ ]:
_LAT_R = math.radians(LEO_LAT_DEG)
_SIN_LAT = math.sin(_LAT_R)
_COS_LAT = math.cos(_LAT_R)

def _gmst_hours(unix_t):
    jd = unix_t / 86400.0 + 2440587.5
    d = jd - 2451545.0
    return (18.697374558 + 24.06570982441908 * d) % 24.0

def fast_altaz(ra_deg, dec_deg, unix_t):
    lst_deg = (_gmst_hours(unix_t) * 15.0 + LEO_LON_DEG) % 360.0
    ha = math.radians(((lst_deg - ra_deg + 180.0) % 360.0) - 180.0)
    dec = math.radians(dec_deg)
    sin_alt = max(-1.0, min(1.0,
        _SIN_LAT * math.sin(dec) + _COS_LAT * math.cos(dec) * math.cos(ha)))
    alt = math.degrees(math.asin(sin_alt))
    cos_alt = math.sqrt(max(0.0, 1.0 - sin_alt * sin_alt))
    if cos_alt < 1e-9:
        return alt, 0.0
    sin_az = -math.cos(dec) * math.sin(ha) / cos_alt
    cos_az = (math.sin(dec) - _SIN_LAT * sin_alt) / (_COS_LAT * cos_alt)
    return alt, (math.degrees(math.atan2(sin_az, cos_az)) + 360.0) % 360.0

def cell_radec(l, b):
    gc = SkyCoord(l=l * u.deg, b=b * u.deg, frame='galactic')
    return float(gc.icrs.ra.deg), float(gc.icrs.dec.deg)

def forward_simulate(cells, t0, cell_time_sec=CELL_TIME_P50,
                     recal_block_sec=RECAL_BLOCK_SEC,
                     recal_every_n=RECAL_EVERY_N,
                     horizon_h=MAX_PLAN_HORIZON_H):
    """Project alt/az at each cell's expected observation time."""
    rows = []
    horizon_s = horizon_h * 3600.0
    for i, (_, _, l, b) in enumerate(cells):
        n_recals = 1 + i // recal_every_n
        offset = (i + 0.5) * cell_time_sec + n_recals * recal_block_sec
        ra, dec = cell_radec(l, b)
        alt, az = fast_altaz(ra, dec, t0 + offset)
        in_alt = MIN_ALT_DEG <= alt <= MAX_ALT_DEG
        in_az = AZ_MIN_DEG <= az <= AZ_MAX_DEG
        in_horizon = offset <= horizon_s
        rows.append({
            'l': l, 'b': b, 'offset_s': offset,
            'alt': alt, 'az': az,
            'kept': in_alt and in_az and in_horizon,
        })
    return rows

# A quick "what would a session look like *right now*?" sanity demo.
now = time.time()
sim = forward_simulate(even_grid, now)
n_kept = sum(r['kept'] for r in sim)
print(f'forward sim @ now: {n_kept}/{len(sim)} cells kept '
      f'({100 * n_kept / max(len(sim), 1):.0f}%)')

## 5. Reconstructing archived sessions

Each on-disk session in `data/archive/main/session_NNN/` is a directory of cell subdirectories (`obs_<l>_<b>` and `cal_<l>_<b>`), each containing `.npz` dumps whose filenames encode UTC timestamps as `_YYYYMMDD_HHMMSS.npz`. Reconstructing the scan order is just sorting cells by the earliest dump timestamp within each cell directory. The cell's *observed alt/az* is then taken at that first-dump time, which is what the dish was actually pointed at (modulo a few-second slew settle, negligible for visualisation).

We index sessions by their coverage span -- the number of distinct cells and the $\ell$ range -- so we can pick a couple of "comprehensive" examples to plot.

In [ ]:
_DUMP_RE = re.compile(r'^(?:obs|cal)_(.+?)_(-?\d+)_(\d{8}_\d{6})\.npz$')
_OBSCAL_RE = re.compile(r'^(obs|cal)_(.+?)_(-?\d+)$')

def _parse_lb(name_l):
    return float(name_l.replace('p', '.'))

def load_session(session_dir):
    """Return ordered list of dicts: one entry per science (obs) cell.

    The cell's observation time is the earliest dump timestamp in that
    cell's obs_<l>_<b>/ directory.  Recal pointings (cal_<ra>_<dec> with
    integer or non-grid labels) are skipped here -- they appear as a
    second list returned alongside.
    """
    cells = {}
    recals = {}
    session_dir = Path(session_dir)
    for sub in session_dir.iterdir():
        if not sub.is_dir():
            continue
        m = _OBSCAL_RE.match(sub.name)
        if m is None:
            continue
        kind, l_name, b_str = m.group(1), m.group(2), m.group(3)
        # Pull all dump timestamps in this dir.
        ts = []
        for f in sub.glob('*.npz'):
            mm = _DUMP_RE.match(f.name)
            if mm is None:
                continue
            try:
                ts.append(time.mktime(time.strptime(mm.group(3), '%Y%m%d_%H%M%S')))
            except ValueError:
                pass
        if not ts:
            continue
        t_first = min(ts)
        try:
            l = _parse_lb(l_name)
        except ValueError:
            continue
        b = int(b_str)
        bucket = recals if kind == 'cal' and (abs(b) > B_MAX or abs(l) > 360) else cells
        # Treat anything obs_ as a science cell.  cal_ on the survey grid
        # is a calibration dump for the same pointing, not a recal field;
        # we merge by (l, b) and keep the earliest time.
        if kind == 'obs' or (kind == 'cal' and (l, b) in bucket):
            key = (l, b)
            entry = bucket.get(key)
            if entry is None or t_first < entry['t']:
                bucket[key] = {'l': l, 'b': b, 't': t_first}
        elif kind == 'cal':
            # Cal at a non-grid pointing -> a recal field.
            recals[(l, b)] = {'l': l, 'b': b, 't': t_first}
    science = sorted(cells.values(), key=lambda r: r['t'])
    recal_list = sorted(recals.values(), key=lambda r: r['t'])
    return science, recal_list

ARCHIVE = Path('data/archive/main')
session_dirs = sorted(p for p in ARCHIVE.glob('session_*') if p.is_dir())

sessions = []
for sd in session_dirs:
    science, recals = load_session(sd)
    if not science:
        continue
    t_start = science[0]['t']
    t_end = science[-1]['t']
    ls = [c['l'] for c in science]
    sessions.append({
        'name': sd.name,
        'cells': science,
        'recals': recals,
        't_start': t_start,
        't_end': t_end,
        'duration_h': (t_end - t_start) / 3600.0,
        'n_cells': len(science),
        'l_span': max(ls) - min(ls) if ls else 0.0,
    })

sessions.sort(key=lambda s: (s['n_cells'], s['l_span']), reverse=True)
print(f'{len(sessions)} sessions loaded from {ARCHIVE}')
print()
print(f'{"session":<14}{"cells":>7}{"l span":>10}{"hours":>8}')
for s in sessions[:8]:
    print(f'  {s["name"]:<12}{s["n_cells"]:>7}{s["l_span"]:>10.0f}'
          f'{s["duration_h"]:>8.1f}')

## 6. Representative sessions in galactic coordinates

We pick the top-$N$ archived sessions by cell count and plot their footprints in flat-sky $(\ell, b)$. The full-grid layout is shown faintly in the background; each session's cells are coloured by their scan time within the session. The plot makes plain how a single session sweeps a $\sim 200^\circ$ arc of the plane while the rest is pushed to other nights' sessions.

In [ ]:
TOP_N = 2
top_sessions = sessions[:TOP_N]

grid_l = np.array([c[2] for c in even_grid])
grid_b = np.array([c[3] for c in even_grid])

fig, axes = plt.subplots(
    len(top_sessions), 1,
    figsize=(plotting.TEXTWIDTH_IN, 2.4 * len(top_sessions)),
    sharex=True,
)
if len(top_sessions) == 1:
    axes = [axes]

for ax, s in zip(axes, top_sessions):
    ax.scatter(grid_l, grid_b, c='lightgrey', s=plotting.SS_FINE * 0.5,
               edgecolors='none', alpha=0.5, zorder=1, label='full grid')
    ls = np.array([c['l'] for c in s['cells']])
    bs = np.array([c['b'] for c in s['cells']])
    ts = np.array([c['t'] for c in s['cells']])
    hours = (ts - s['t_start']) / 3600.0
    sc = ax.scatter(ls, bs, c=hours, cmap='viridis', s=plotting.SS_FINE * 1.6,
                    edgecolors='none', zorder=3,
                    label=f'{s["n_cells"]} cells')
    cbar = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.01)
    cbar.set_label('hours into session', fontsize=plotting.TICK_SIZE - 2)
    ax.set_xlim(L_MAX + 5, L_MIN - 5)
    ax.set_ylim(B_MIN - 3, B_MAX + 3)
    ax.set_aspect('equal')
    ax.set_ylabel(r'$b$ [deg]', fontsize=plotting.LABEL_SIZE)
    ax.set_title(
        rf'{s["name"]}: {s["n_cells"]} cells, '
        rf'{s["duration_h"]:.1f} h, $\ell$ span {s["l_span"]:.0f}$^\circ$',
        fontsize=plotting.LABEL_SIZE,
    )
    ax.grid(True, **plotting.GRID_STYLE)
    ax.legend(fontsize=plotting.LEGEND_SIZE - 1, loc='lower left')

axes[-1].set_xlabel(r'$\ell$ [deg]', fontsize=plotting.LABEL_SIZE)
fig.tight_layout()
plt.show()

## 7. Representative sessions in topocentric coordinates

The same sessions plotted on a topocentric Mollweide projection make the dish's accessibility cuts visible directly. Each panel shows:

* the dish's inaccessible regions (red hatching, alt cuts at $17^\circ$ and $83^\circ$ plus the cable-wrap wedge around az $= 0/360^\circ$);
* one beam-sized circle ($\mathrm{HPBW} = 3.4^\circ$, with $1/\cos(\mathrm{alt})$ longitude correction near the zenith) per observed cell, coloured by scan time;
* a faint great-circle-on-the-sphere track threading the cells in observation order, so consecutive slews are visible.

The session's az-side selection becomes obvious in this view: every panel sits cleanly on one hemisphere (the planner picked rising or setting based on which side held more cells at $t_0$), and the cell track never crosses the cable-wrap wedge.

In [ ]:
def _az_to_lon(az_deg):
    """Map az in [0, 360) to lon in [-180, 180) with az=0 at lon=0."""
    az = np.asarray(az_deg, dtype=float) % 360.0
    return np.where(az <= 180.0, az, az - 360.0)

fig, axes = plt.subplots(
    len(top_sessions), 1,
    figsize=(plotting.TEXTWIDTH_IN, 4.0 * len(top_sessions)),
    subplot_kw={'projection': 'mollweide'},
)
if len(top_sessions) == 1:
    axes = [axes]

for ax, s in zip(axes, top_sessions):
    ax.grid(True, **{k: v for k, v in plotting.GRID_STYLE.items() if k != 'color'},
            color=plotting.NEUTRAL_COLOR)
    add_topo_inaccessible_overlay(ax)

    alts = []
    azs = []
    times = []
    for c in s['cells']:
        ra, dec = cell_radec(c['l'], c['b'])
        alt, az = fast_altaz(ra, dec, c['t'])
        alts.append(alt)
        azs.append(az)
        times.append(c['t'])
    alts = np.array(alts)
    azs = np.array(azs)
    lons = _az_to_lon(azs)
    hours = (np.array(times) - s['t_start']) / 3600.0

    # Beam circles (one polygon per cell, colored by scan time).
    polys, src_idx = beam_outline_polys(
        lons, alts, HPBW_DEG,
        n_vertices=33, lat_clamp_deg=75.0, seam_lon_deg=180.0,
    )
    cmap = plt.get_cmap('viridis')
    norm = mpl.colors.Normalize(vmin=hours.min(), vmax=hours.max())
    colors = cmap(norm(hours[src_idx]))
    pc = PolyCollection(polys, facecolors=colors,
                        edgecolors='black', linewidths=0.2,
                        alpha=0.85, zorder=4)
    ax.add_collection(pc)

    # Connecting track, broken at the +/-180 seam.
    pts = np.column_stack([np.deg2rad(lons), np.deg2rad(alts)])
    segs = []
    for p0, p1 in zip(pts[:-1], pts[1:]):
        if abs(p1[0] - p0[0]) > math.pi:
            continue
        segs.append([p0, p1])
    lc = LineCollection(segs, colors='k', linewidths=plotting.LW_FINE,
                        alpha=0.35, zorder=3)
    ax.add_collection(lc)

    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.7, pad=0.05)
    cbar.set_label('hours into session', fontsize=plotting.TICK_SIZE - 2)

    az_ticks = np.array([-150, -120, -90, -60, -30, 0, 30, 60, 90, 120, 150])
    ax.set_xticks(np.deg2rad(az_ticks))
    ax.set_xticklabels(
        [rf'{int(t % 360)}$^\circ$' for t in az_ticks],
        fontsize=plotting.TICK_SIZE - 3,
    )
    ax.set_title(
        rf'{s["name"]}: topocentric, az$=0$ centred, '
        rf'{s["n_cells"]} cells',
        fontsize=plotting.LABEL_SIZE,
    )

fig.tight_layout()
plt.show()

## 8. Aggregate coverage across all archived sessions

Stacking the cells from every archived session shows how the per-session bites add up to the full survey. Each unique grid cell is counted by the number of distinct sessions that visited it; the colormap therefore doubles as a *redundancy* indicator. Cells visited many times have stronger statistics (used in `main_scan_load_diagnostics.ipynb` for cross-session drift checks); cells visited once or not at all are next-pass priorities.

In [ ]:
visits = defaultdict(int)
for s in sessions:
    seen = set()
    for c in s['cells']:
        key = (c['l'], c['b'])
        if key not in seen:
            visits[key] += 1
            seen.add(key)

grid_set = {(round(l, 2), b) for _, _, l, b in even_grid}
covered_keys = set(visits) & grid_set
n_covered = len(covered_keys)
n_grid = len(grid_set)
print(f'unique grid cells observed: {n_covered} / {n_grid} '
      f'({100 * n_covered / n_grid:.0f}%)')
if visits:
    max_visits = max(visits.values())
    print(f'max revisits on any cell:  {max_visits}')
    print(f'median revisits (covered): '
          f'{int(np.median([v for k, v in visits.items() if k in grid_set])):d}')

fig, ax = plt.subplots(figsize=(plotting.TEXTWIDTH_IN, 3.2))
ax.scatter(grid_l, grid_b, c='lightgrey', s=plotting.SS_FINE * 0.5,
           edgecolors='none', alpha=0.4, zorder=1, label='unvisited')
if visits:
    keys = sorted(covered_keys)
    ls = np.array([k[0] for k in keys])
    bs = np.array([k[1] for k in keys])
    cs = np.array([visits[k] for k in keys])
    sc = ax.scatter(ls, bs, c=cs, cmap='magma',
                    s=plotting.SS_FINE * 1.4, edgecolors='none', zorder=3)
    cbar = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.01)
    cbar.set_label('sessions observing this cell',
                   fontsize=plotting.TICK_SIZE - 2)

ax.set_xlim(L_MAX + 5, L_MIN - 5)
ax.set_ylim(B_MIN - 3, B_MAX + 3)
ax.set_aspect('equal')
ax.set_xlabel(r'$\ell$ [deg]', fontsize=plotting.LABEL_SIZE)
ax.set_ylabel(r'$b$ [deg]', fontsize=plotting.LABEL_SIZE)
ax.set_title(
    f'cumulative coverage across {len(sessions)} archived sessions',
    fontsize=plotting.LABEL_SIZE,
)
ax.grid(True, **plotting.GRID_STYLE)
ax.legend(fontsize=plotting.LEGEND_SIZE - 1, loc='lower left')
fig.tight_layout()
plt.show()